# Train Graph-DiT on QM9 (Molecule Scout V4)

Run on Google Colab with a free T4: **Runtime > Change runtime type > T4 GPU**. Produces `graphdit-qm9.pt` + provenance JSON. Download both, place the `.pt` at `backend/models/graphdit-qm9.pt`, and report back the provenance + validity number.

~1-3h on T4 for 100 epochs. Run all cells top to bottom.

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a T4 GPU: Runtime > Change runtime type"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install -q torch-molecule

In [ ]:
from torch_molecule.datasets import load_qm9

dataset, _ = load_qm9(target_cols=["gap"], return_local_data_path=True)
smiles = list(dataset.data)
print(len(smiles), smiles[:3])

In [ ]:
from torch_molecule.generator.graph_dit.modeling_graph_dit import (
    GraphDITMolecularGenerator,
)

EPOCHS = 100
model = GraphDITMolecularGenerator(
    num_layer=6,
    hidden_size=512,
    num_head=8,
    timesteps=500,
    batch_size=128,
    epochs=EPOCHS,
    learning_rate=2e-4,
    verbose="print_statement",
    device="cuda",
)
model.fit(smiles)

In [ ]:
batch = model.generate(batch_size=32)

from rdkit import Chem

valid = [s for s in batch if s and Chem.MolFromSmiles(s)]
print(f"validity: {len(valid)}/{len(batch)} = {len(valid) / len(batch):.3f} (expect > 0.9)")
print(valid[:10])

In [ ]:
import hashlib
import json

OUT = "graphdit-qm9.pt"
model.save_to_local(OUT)
with open(OUT, "rb") as f:
    sha = hashlib.sha256(f.read()).hexdigest()[:12]
provenance = {
    "model": "GraphDITMolecularGenerator",
    "dataset": "QM9 (liuganghuggingface/QM9)",
    "n_train": len(smiles),
    "epochs": EPOCHS,
    "checkpoint_sha": sha,
}
with open("graphdit-qm9.provenance.json", "w") as f:
    json.dump(provenance, f, indent=2)
print(provenance)

## Next steps

1. Download `graphdit-qm9.pt` + `graphdit-qm9.provenance.json` (file browser, right-click > Download).
2. Place the `.pt` at `backend/models/graphdit-qm9.pt` in the repo (gitignored).
3. Report back: the provenance JSON + the validity number from the sanity cell.